# Step 10b — MVT Grid: CIFAR-FS, 1-shot

**Settings:** GPU T4, Internet ON, attach the Kaggle dataset `beft-thesis-data`
(owner `notavailable73` — note the slug is missing the "p" in "bpeft", a
pre-existing typo in how the dataset was created; the display title still
reads "bpeft-thesis-data"). See `step_writeups/step10.txt` for the full
reasoning and `plan.md` for the grid design. This notebook only drives
`scripts/run_mvt_grid.py --only "dataset=cifar_fs,shots=1"`; every other Step 10 script (config
generation, aggregation, tables, plots) is shared across all three notebooks
and documented there — nothing new lives in this notebook itself.


## 1. GPU check + clone repo + install deps

In [1]:
import torch, sys, os, subprocess
print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU T4'

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/thesis'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repo ready at', os.getcwd())


python: 3.12.13 | torch: 2.10.0+cu128
cuda  : True | Tesla T4


Cloning into '/kaggle/working/thesis'...


repo ready at /kaggle/working/thesis


## 2. Stage data — symlink the attached dataset into `data/`

Nothing here is a hard requirement: every dataset falls back to a runtime download if its symlink source is missing (Internet must be ON).

In [2]:
# Attach the Kaggle dataset `beft-thesis-data` (owner notavailable73) before
# running this cell: https://www.kaggle.com/datasets/notavailable73/beft-thesis-data
# Its real upload structure (verified via the Kaggle API, 2026-08-03) is:
#   bpeft-data/cifar-100-python/{meta,train,test}
#   bpeft-data/svhn/test_32x32.mat
#   bpeft-data/tinyimagenet/tiny-imagenet-200/tiny-imagenet-200/{train,val,test,wnids.txt,...}
#   bpeft-data/miniimagenet/mini-imagenet-cache-{train,validation,test}.pkl
# (Zenodo pkl caches, not the .npy/.json format plan.md Section 4.2 originally
# assumed -- src/datasets/mini_imagenet.py already supports this layout
# natively, so nothing needed regenerating.) Rather than hardcode that exact
# nesting, this reuses the SAME staged-path finder functions
# scripts/train.py / evaluate.py call at runtime, so it is guaranteed to
# symlink to whatever those modules would discover themselves -- correct
# regardless of whether Kaggle mounts this dataset one level deeper
# (`/kaggle/input/datasets/<owner>/<slug>/...`) or double-wraps the
# tiny-imagenet-200 folder on auto-unzip (both observed live in earlier
# steps; see each finder's docstring).
import os, shutil

from src.datasets.cifar_fs import _find_staged_cifar100_root
from src.datasets.svhn_ood import _find_staged_svhn_root
from src.datasets.tinyimagenet_ood import _find_extracted_tin_root
from src.datasets.mini_imagenet import _find_zenodo_pkls

LINKS = {}

cifar100_root = _find_staged_cifar100_root('data')
if cifar100_root:
    LINKS['data/cifar-100-python'] = os.path.join(cifar100_root, 'cifar-100-python')

svhn_root = _find_staged_svhn_root('data')
if svhn_root:
    LINKS['data/svhn/test_32x32.mat'] = os.path.join(svhn_root, 'test_32x32.mat')

tin_root = _find_extracted_tin_root('data')
if tin_root:
    LINKS['data/tiny-imagenet-200'] = tin_root

for split, pkl_path in (_find_zenodo_pkls('data') or {}).items():
    LINKS[f'data/{pkl_path.name}'] = str(pkl_path)

os.makedirs('data/svhn', exist_ok=True)
if not LINKS:
    print('No staged files found under /kaggle/input -- did you attach '
          '`beft-thesis-data`? Falling back to runtime downloads for everything.')
for link, target in LINKS.items():
    if os.path.exists(link) or os.path.islink(link):
        print(f'OK   (already present): {link}')
        continue
    if not os.path.exists(target):
        print(f'MISSING source -- {os.path.basename(link)} will fall back to '
             f'runtime download (target not found: {target})')
        continue
    try:
        os.symlink(target, link)
        print(f'OK   (symlinked): {link} -> {target}')
    except OSError as e:
        print(f'symlink failed ({e}); copying instead (slower): {link}')
        (shutil.copytree if os.path.isdir(target) else shutil.copy2)(target, link)
        print(f'OK   (copied): {link}')


OK   (symlinked): data/cifar-100-python -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/cifar-100-python
OK   (symlinked): data/svhn/test_32x32.mat -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/svhn/test_32x32.mat
OK   (symlinked): data/tiny-imagenet-200 -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/tinyimagenet/tiny-imagenet-200/tiny-imagenet-200
OK   (symlinked): data/mini-imagenet-cache-train.pkl -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/miniimagenet/mini-imagenet-cache-train.pkl
OK   (symlinked): data/mini-imagenet-cache-validation.pkl -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/miniimagenet/mini-imagenet-cache-validation.pkl
OK   (symlinked): data/mini-imagenet-cache-test.pkl -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/miniimagenet/mini-imagenet-cache-test.pkl


## 3. Build the frozen splits (CIFAR-FS + MiniImageNet)

In [3]:
!python scripts/build_cifar_fs_split.py
!python scripts/build_mini_imagenet_split.py


wrote /kaggle/working/thesis/data/cifar_fs_split.json  (64/16/20, disjoint, union=100, status=canonical_bertinetto_via_torchmeta)
wrote /kaggle/working/thesis/data/mini_imagenet_split.json  (64/16/20, disjoint, union=100, status=canonical_ravi_larochelle)


## 4. Generate the 120 grid configs + run the offline config tests

In [4]:
!python scripts/build_grid_configs.py
!python -m pytest -q tests/test_grid_configs.py


wrote 120 configs to /kaggle/working/thesis/configs/grid/ (96 PEFT + 24 baseline)
wrote /kaggle/working/thesis/configs/grid/_index.json
  priority 1: 36 cells
  priority 2: 24 cells
  priority 3: 36 cells
  priority 4: 24 cells
...................                                                      [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-pa

## 5. Log in to W&B + run the grid

Add a Kaggle Secret named `WANDB_API_KEY` (notebook editor → Add-ons →
Secrets; get the key from <https://wandb.ai/authorize>) before running this
cell so the grid's runs upload online and group by (dataset, shots) per
`progress.txt`'s Step 10 exit criteria. Falls back to **offline** mode
(writes to `./wandb/`, sync later with `wandb sync wandb/`) if the secret is
missing or login fails — this cell never calls interactive `wandb.login()`,
so the unattended `--max-minutes 660` run below can't stall waiting on
stdin. Login and the grid launch are ONE cell on purpose (a prior version
split them across two cells and passed `WANDB_MODE` to a separate `!`
shell cell via `--wandb-mode {WANDB_MODE}`; that silently broke if the two
cells were ever run out of order or after a kernel restart, since IPython
leaves an unresolved `{name}` in a `!` command as literal text instead of
erroring). Resumable: re-running this cell after a session timeout picks up
where it left off (`--resume` skips any cell whose results JSON already
exists).

In [5]:
import os
import subprocess
import sys

import wandb

api_key = os.environ.get("WANDB_API_KEY")
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if api_key:
            os.environ["WANDB_API_KEY"] = api_key
            print("Loaded WANDB_API_KEY from Kaggle Secrets.")
    except Exception as e:
        print(f"Kaggle Secrets lookup skipped: {e!r}")

if not api_key:
    print("No WANDB_API_KEY found (env or Kaggle Secrets) -- using offline "
          "mode so this run never blocks on an interactive login prompt.")
    WANDB_MODE = "offline"
else:
    WANDB_MODE = "online"
    try:
        if not wandb.login(key=api_key):
            print("wandb.login() returned False -- falling back to offline mode.")
            WANDB_MODE = "offline"
    except Exception as e:
        print(f"wandb.login() failed: {e!r} -- falling back to offline mode.")
        WANDB_MODE = "offline"

print("WANDB_MODE for this session:", WANDB_MODE)

cmd = [sys.executable, "scripts/run_mvt_grid.py",
       "--resume", "--only", "dataset=cifar_fs,shots=1",
       "--max-minutes", "660", "--use-tinyimagenet", "--use-gaussian",
       "--wandb-mode", WANDB_MODE]
print(">>>", " ".join(cmd))
subprocess.run(cmd)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: fatpotato (fatpotato-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded WANDB_API_KEY from Kaggle Secrets.
WANDB_MODE for this session: online
>>> /usr/bin/python3 scripts/run_mvt_grid.py --resume --only dataset=cifar_fs,shots=1 --max-minutes 660 --use-tinyimagenet --use-gaussian --wandb-mode online
[grid] 36 cells selected (filtered by dataset=cifar_fs,shots=1)
[grid] (1/36) cifar_fs/1shot/resnet18/bottleneck_parallel/evidential/seed42
[22:15:53] INFO bpeft.train: config: /kaggle/working/thesis/configs/grid/cifar_1shot_r18_parallel_evidential_seed42.yaml  seed: 42  trainer.type: episodic


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: fatpotato (fatpotato-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/thesis/wandb/run-20260803_221554-1nbaetyw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet18_bottleneck_prototype_cifar_fs_1shot_seed42
wandb: ⭐️ View project at https://wandb.ai/fatpotato-personal/bpeft-thesis
wandb: 🚀 View run at https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/1nbaetyw


[22:15:56] INFO bpeft.train: wandb run: resnet18_bottleneck_prototype_cifar_fs_1shot_seed42  (online)  url=https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/1nbaetyw
[22:18:34] INFO bpeft.train: backbone: resnet18 (feature_dim=512)  adapter: bottleneck/parallel
[22:18:34] INFO bpeft.train: placement sites: layer1.1(64ch), layer2.1(128ch), layer3.1(256ch), layer4.1(512ch)
[22:18:34] INFO bpeft.train: trainable params: 31,746
[22:19:21] INFO bpeft.train: epoch   1/30  train_loss=0.5133  train_acc=0.733  val_loss=0.6638  val_acc=0.692  kl_w=0.010  mean_ev=1.6295  grad_norm=0.7193  global_step=100
[22:20:05] INFO bpeft.train: epoch   2/30  train_loss=0.4805  train_acc=0.741  val_loss=0.6403  val_acc=0.696  kl_w=0.020  mean_ev=1.7323  grad_norm=0.7503  global_step=200
[22:20:51] INFO bpeft.train: epoch   3/30  train_loss=0.4696  train_acc=0.756  val_loss=0.6483  val_acc=0.689  kl_w=0.030  mean_ev=1.7830  grad_norm=0.8341  global_step=300
[22:21:39] INFO bpeft.train: epoch   4/30  train_

wandb: updating run metadata
wandb: uploading history steps 7-8, summary, console lines 12-14
wandb: 
wandb: Run history:
wandb:         train/acc_epoch ▁▂▄▃▃▃█▆▆
wandb: train/adapter_grad_norm ▁▂▃▅▅▇▇██
wandb:             train/epoch ▁▂▃▄▅▅▆▇█
wandb:       train/global_step ▁▂▃▄▅▅▆▇█
wandb:         train/kl_weight ▁▂▃▄▄▅▆▇█
wandb:        train/loss_epoch █▄▃▄▄▄▁▄▄
wandb:     train/mean_evidence ▁▄▆█▇▆▅▃▄
wandb:                 val/acc ▃▄▂█▆▇▄▂▁
wandb:                val/loss █▅▆▃▃▁▂▃▅
wandb: 
wandb: Run summary:
wandb:                n_params 31746
wandb:         train/acc_epoch 0.77293
wandb: train/adapter_grad_norm 1.07245
wandb:      train/best_val_acc 0.71067
wandb:    train/best_val_epoch 4
wandb:             train/epoch 9
wandb:       train/global_step 900
wandb:         train/kl_weight 0.09
wandb:        train/loss_epoch 0.48159
wandb:     train/mean_evidence 1.71475
wandb:                      +3 ...
wandb: 
wandb: 🚀 View run resnet18_bottleneck_prototype_cifar_fs_1shot_seed42

...

wandb: setting up run o0pbyl0g
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/thesis/wandb/run-20260804_090633-o0pbyl0g
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet18_linear_probe_prototype_cifar_fs_1shot_seed44_eval
wandb: ⭐️ View project at https://wandb.ai/fatpotato-personal/bpeft-thesis
wandb: 🚀 View run at https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/o0pbyl0g


[09:06:35] INFO bpeft.evaluate: wandb run: resnet18_linear_probe_prototype_cifar_fs_1shot_seed44_eval  url=https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/o0pbyl0g
[09:08:01] INFO bpeft.evaluate: loaded checkpoint: checkpoints/model_phase2_grid_cifar_1shot_r18_linear_probe_prototype-softmax_seed44.pt  best_val_epoch=0  best_val_acc=0.599
[09:12:15] INFO bpeft.evaluate: OOD pools: svhn_far=(500, 512), cifar100_near=(500, 512), tin_near=(500, 512), gaussian_far=(500, 512)
[09:13:56] INFO bpeft.evaluate: fit temperature on 100 val episodes: T=0.4432
[09:13:56] INFO bpeft.evaluate: ep   0  acc=0.733  F1=0.720  ECE=0.273  Brier=0.502  AUROC[svhn_far/msp]=0.763
[09:13:57] INFO bpeft.evaluate: ep   1  acc=0.827  F1=0.808  ECE=0.407  Brier=0.487  AUROC[svhn_far/msp]=0.529
[09:13:57] INFO bpeft.evaluate: ep   2  acc=0.827  F1=0.824  ECE=0.471  Brier=0.571  AUROC[svhn_far/msp]=0.645
[09:13:57] INFO bpeft.evaluate: ep   3  acc=0.733  F1=0.724  ECE=0.344  Brier=0.550  AUROC[svhn_far/msp]=0.7

wandb: updating run metadata; uploading artifact metrics_grid_cifar_1shot_r18_linear_probe_seed44_linear_probe_prototype-softmax
wandb: uploading artifact metrics_grid_cifar_1shot_r18_linear_probe_seed44_linear_probe_prototype-softmax
wandb: uploading media/images/plots/reliability_diagram_600_7f635c6f34d62d4a773c.png; uploading media/images/plots/ood_histogram_601_7c59b73029d251a659a9.png; uploading media/images/plots/confusion_matrix_602_9716c848315dfb86ec95.png; uploading output.log; uploading wandb-summary.json (+ 1 more)
wandb: 
wandb: Run history:
wandb:                   eval/ECE ▆▄▆▅▂▄▅▅▇▄▇█▃▆▇▅▅▄▃▅▂▆█▃▃▃▁▅▄▄▇▄▃▄▄▅▄▄▇▄
wandb:              eval/accuracy ▇▃▅█▇▆▆▄▆▆▇▆▄▅█▇▆▆▆█▇▇▆▇▇▂▁▇▇▇▇▄▃▆▆▄▇▄▆▆
wandb: eval/accuracy_running_mean █▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:               eval/episode ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▇▇████
wandb: 
wandb: Run summary:
wandb:                   eval/ECE 0.34415
wandb:              eval/accuracy 0.73333
wandb: eval/accuracy_running

{
  "accuracy_ci95": 0.008350035073779826,
  "accuracy_mean": 0.7025111271440982,
  "accuracy_std": 0.10435370033216053,
  "adapter_type": "linear_probe",
  "best_val_epoch": 0,
  "brier_mean": 0.5282634117702643,
  "brier_std": 0.06755484330278998,
  "brier_ts": 0.4164086878299713,
  "config_path": "/kaggle/working/thesis/configs/grid/cifar_1shot_r18_linear_probe_softmax_seed44.yaml",
  "ece_per_episode_mean": 0.2943684964030981,
  "ece_per_episode_std": 0.08182767950216883,
  "ece_pooled": 0.28182319283220497,
  "ece_ts": 0.06421210902399486,
  "episodes_file": "configs/test_episodes.yaml",
  "f1_macro_ci95": 0.008898919339618236,
  "f1_macro_mean": 0.6910346837254165,
  "f1_macro_std": 0.11121332471555981,
  "fpr_at_95_tpr__cifar100_near__energy": 0.8779666666666667,
  "fpr_at_95_tpr__cifar100_near__msp": 0.8888533333333332,
  "fpr_at_95_tpr__cifar100_near__ts_msp": 0.8922499999999999,
  "fpr_at_95_tpr__gaussian_far__energy": 0.0,
  "fpr_at_95_tpr__gaussian_far__msp": 0.775046666666

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_mvt_grid.py', '--resume', '--only', 'dataset=cifar_fs,shots=1', '--max-minutes', '660', '--use-tinyimagenet', '--use-gaussian', '--wandb-mode', 'online'], returncode=0)

## 6. Pack + push this notebook's slice of results

In [6]:
# Pack + push this notebook's slice only (mirrors Step 9 notebook Section 9/9b).
import glob, hashlib, json as _json, os, subprocess, zipfile

ARTIFACT_STEM = 'step10b_cifar_1shot'
RUN_TAG_GLOB = 'grid_cifar_1shot'

ZIP = f'/kaggle/working/{ARTIFACT_STEM}_artifacts.zip'
RESULTS = sorted(glob.glob(f'results/grid/*{RUN_TAG_GLOB}*'))
LOGS = ['results/grid/_run_log.jsonl'] if os.path.exists('results/grid/_run_log.jsonl') else []
CHECKPOINTS = sorted(glob.glob(f'checkpoints/model_phase2_{RUN_TAG_GLOB}*.pt'))
ALL_FILES = RESULTS + LOGS + CHECKPOINTS

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in ALL_FILES:
        zf.write(p)
    manifest_lines = [
        f'{p}  {os.path.getsize(p)}B  sha256={hashlib.sha256(open(p, "rb").read()).hexdigest()}'
        for p in ALL_FILES
    ]
    zf.writestr('MANIFEST.txt', '\n'.join(manifest_lines))
print(f'wrote {ZIP} ({len(ALL_FILES)} files)')

# Channel 1: push to a Kaggle dataset (survives the browser tab closing) if
# KAGGLE_USERNAME / KAGGLE_KEY secrets are configured; channel 2: browser
# download otherwise. See notebooks/step9-mini(1).ipynb Section 9b for the
# fuller version of this pattern this is condensed from.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')
    HAVE_SECRETS = True
except Exception:
    HAVE_SECRETS = False

if HAVE_SECRETS:
    ds_dir = f'/kaggle/working/{ARTIFACT_STEM}_dataset'
    os.makedirs(ds_dir, exist_ok=True)
    subprocess.run(['cp', ZIP, ds_dir], check=True)
    meta = {
        "title": f"{ARTIFACT_STEM}-artifacts",
        "id": f"{os.environ['KAGGLE_USERNAME']}/{ARTIFACT_STEM}-artifacts",
        "licenses": [{"name": "CC0-1.0"}],
    }
    _json.dump(meta, open(f'{ds_dir}/dataset-metadata.json', 'w'))
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', ds_dir, '-q'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run(['kaggle', 'datasets', 'version', '-p', ds_dir, '-m', 'update', '-q'])
    print(f'pushed to Kaggle dataset {ARTIFACT_STEM}-artifacts')
else:
    from IPython.display import FileLink, display
    display(FileLink(ZIP))
    print('no Kaggle Secrets found -- use the download link above instead.')


wrote /kaggle/working/step10b_cifar_1shot_artifacts.zip (181 files)
pushed to Kaggle dataset step10b_cifar_1shot-artifacts


## After this session

1. Do **not** run `scripts/aggregate_grid.py` / `make_master_tables.py` /
   `grid_plots.py` here — those run ONCE, locally or on plain CPU, after
   10a + 10b + 10c have ALL finished and their artifact zips are merged into
   one local `results/grid/` (plan.md Section 10.4).
2. Note any `collapsed` or `error` status lines from Section 6's output, and
   `results/grid/_run_log.jsonl`'s `best_val_epoch` values, for
   `step_writeups/step10.txt`.
